# ProntoQA Symbolic Pipeline

End-to-end pipeline using real ProntoQA problems from HuggingFace:
1. Config
2. Mount Google Drive
3. Clone repo + install deps
4. Load ProntoQA + generate Qwen traces
5. Train SSAE Phase 1
6. Generate probe data (encode traces through SSAE)
7. Train step-correctness probe
8. Download results

## 0 — Config

In [1]:
# ── Edit these ────────────────────────────────────────────────────────────────
GITHUB_REPO   = "https://github.com/djaxchi/CoT-checker.git"

# Data generation
# ProntoQA has 500 problems (~48% False) — None = use all, or set e.g. 50 for a smoke test
MAX_PROBLEMS  = None
GEN_BATCH     = 16      # LLM generation batch size

# SSAE training
TRAIN_EPOCHS  = 1
TRAIN_BATCH   = 16
GRAD_ACCUM    = 4       # effective batch = 16 * 4 = 64

# Probe training
PROBE_EPOCHS  = 30
# ── End config ────────────────────────────────────────────────────────────────

import torch
gpu  = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (no GPU!)"
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f"GPU : {gpu}")
print(f"VRAM: {vram:.0f} GB")

FREEZE_BACKBONES = vram < 35
print(f"Freeze backbones: {FREEZE_BACKBONES}  ({'T4/V100' if FREEZE_BACKBONES else 'A100'})"
)

GPU : Tesla T4
VRAM: 16 GB
Freeze backbones: True  (T4/V100)


## 1 — Mount Google Drive

Everything is persisted to `MyDrive/CoT-checker/` so runtime resets don't lose progress.

In [5]:
import os
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT        = "/content/drive/MyDrive/CoT-checker"
DRIVE_DATA        = f"{DRIVE_ROOT}/data"
DRIVE_CKPT        = f"{DRIVE_ROOT}/results/checkpoints"
DRIVE_PROBE_DATA  = f"{DRIVE_ROOT}/results/probe_data"
DRIVE_PROBE_MODEL = f"{DRIVE_ROOT}/results/probes"

for d in [DRIVE_DATA, DRIVE_CKPT, DRIVE_PROBE_DATA, DRIVE_PROBE_MODEL]:
    os.makedirs(d, exist_ok=True)

PROBLEMS_PATH    = f"{DRIVE_DATA}/prontoqa_hf.jsonl"
TRACES_PATH      = f"{DRIVE_DATA}/prontoqa_traces.jsonl"
CKPT_PATH        = f"{DRIVE_CKPT}/ssae_symbolic_p1.pt"
ENC_PATH         = f"{DRIVE_CKPT}/ssae_symbolic_p1.enc.pt"
PROBE_DATA_PATH  = f"{DRIVE_PROBE_DATA}/symbolic_ssae_p1.npz"
PROBE_MODEL_PATH = f"{DRIVE_PROBE_MODEL}/correctness_probe_symbolic.pt"

print(f"Traces on Drive    : {'YES - skipping generation' if os.path.exists(TRACES_PATH) else 'NO - will generate'}")
print(f"SSAE ckpt on Drive : {'YES - skipping training'   if os.path.exists(CKPT_PATH)   else 'NO - will train'}")
print(f"Probe data on Drive: {'YES - skipping encoding'   if os.path.exists(PROBE_DATA_PATH) else 'NO - will encode'}")
print(f"Probe model on Drive: {'YES' if os.path.exists(PROBE_MODEL_PATH) else 'NO'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Traces on Drive    : NO - will generate
SSAE ckpt on Drive : NO - will train
Probe data on Drive: NO - will encode
Probe model on Drive: NO


## 2 — Clone repo + install dependencies

In [6]:
import os

if not os.path.isdir("/content/CoT-checker"):
    !git clone {GITHUB_REPO} /content/CoT-checker
else:
    !git -C /content/CoT-checker pull

%cd /content/CoT-checker
!pip install -q transformers datasets tqdm

Cloning into '/content/CoT-checker'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 82 (delta 20), reused 80 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 1.03 MiB | 4.09 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/CoT-checker


## 3 — Load ProntoQA + generate model traces

**Phase A**: loads 500 problems from `renma/ProntoQA` on HuggingFace (~48% False, complex multi-hop chains).  
**Phase B**: runs Qwen2.5-0.5B on each problem to produce model-generated CoT traces.

Skipped automatically if traces already exist on Drive.

In [7]:
import os

if os.path.exists(TRACES_PATH):
    print(f"Traces already exist at {TRACES_PATH} — skipping generation.")
else:
    max_flag = f"--max-problems {MAX_PROBLEMS}" if MAX_PROBLEMS else ""
    !python scripts/generate_symbolic_data.py \
        {max_flag} \
        --problems-out {PROBLEMS_PATH} \
        --traces-out   {TRACES_PATH} \
        --batch-size   {GEN_BATCH} \
        --device cuda
    print("\nDone. Saved to Drive.")

Generating 14000 synthetic problems …
Hop distribution: {1: 2812, 2: 4853, 3: 4264, 4: 2071}
Saved 14000 problems to /content/drive/MyDrive/CoT-checker/data/prontoqa_hf.jsonl

Loading Qwen2.5-0.5B on cuda …
config.json: 100% 681/681 [00:00<00:00, 3.38MB/s]
tokenizer_config.json: 7.23kB [00:00, 22.4MB/s]
vocab.json: 2.78MB [00:00, 69.5MB/s]
merges.txt: 1.67MB [00:00, 118MB/s]
tokenizer.json: 7.03MB [00:00, 144MB/s]
model.safetensors: 100% 988M/988M [00:10<00:00, 96.6MB/s]
Loading weights: 100% 290/290 [00:00<00:00, 377.76it/s, Materializing param=model.norm.weight]                              
generation_config.json: 100% 138/138 [00:00<00:00, 958kB/s]
Generating traces: 100% 875/875 [37:01<00:00,  2.54s/it]

Problems with traces : 14000/14000
Skipped (no steps)   : 0
Total steps          : 45273
Avg steps/problem    : 3.2
Saved 14000 traces (45273 steps) to /content/drive/MyDrive/CoT-checker/data/prontoqa_traces.jsonl

Done. Saved to Drive.


In [ ]:
# Sanity check: inspect traces
import json

total_traces = sum(1 for _ in open(TRACES_PATH))
total_steps  = sum(len(json.loads(l)["steps"]) for l in open(TRACES_PATH))
print(f"Traces : {total_traces}")
print(f"Steps  : {total_steps}")
print(f"Avg steps/trace: {total_steps/total_traces:.1f}")
print()
with open(TRACES_PATH) as f:
    for rec in [json.loads(next(f)) for _ in range(3)]:
        print(f"Q: {rec['question'][:120]}")
        print(f"Steps: {rec['steps']}")
        print()

## 4 — Train SSAE Phase 1

Trains the sparse autoencoder to reconstruct reasoning steps from sparse latents.  
Backbone frozen on T4 (16 GB), unfrozen on A100 (40 GB).  
Checkpoint saved to Drive — safe to interrupt and resume.

In [ ]:
import os

if os.path.exists(CKPT_PATH):
    print(f"Checkpoint already exists at {CKPT_PATH} — skipping SSAE training.")
else:
    freeze_flag = "--freeze-backbones" if FREEZE_BACKBONES else "--no-freeze-backbones"
    !python scripts/train_ssae_symbolic.py \
        --traces    {TRACES_PATH} \
        --output    {CKPT_PATH} \
        --phase     1 \
        --epochs    {TRAIN_EPOCHS} \
        --batch-size {TRAIN_BATCH} \
        --grad-accum {GRAD_ACCUM} \
        --device    cuda \
        {freeze_flag}

In [ ]:
# Verify checkpoint on Drive
import os
for fname in os.listdir(DRIVE_CKPT):
    size = os.path.getsize(f"{DRIVE_CKPT}/{fname}") / 1e6
    print(f"{fname:<50}  {size:6.1f} MB")

## 5 — Generate probe data

Encodes every `(context, step)` pair through the SSAE encoder → sparse latent `h_c`.  
Labels each step with `PropLogicSolver` (1 = valid deduction, 0 = invalid).  
Saves `(latents, labels)` as `.npz`.

**Check the label distribution here** — target is ~25–45% incorrect steps.

In [ ]:
import os

if os.path.exists(PROBE_DATA_PATH):
    print(f"Probe data already exists at {PROBE_DATA_PATH} — skipping encoding.")
else:
    !python scripts/generate_probe_data_symbolic.py \
        --checkpoint {CKPT_PATH} \
        --data       {PROBLEMS_PATH} \
        --output     {PROBE_DATA_PATH} \
        --device     cuda

# Report label distribution
import numpy as np
d = np.load(PROBE_DATA_PATH)
labels = d["correctness"]
pos = int(labels.sum())
neg = len(labels) - pos
print(f"\nProbe data: {len(labels)} steps")
print(f"  Correct (+)  : {pos}  ({pos/len(labels):.1%})")
print(f"  Incorrect (-): {neg}  ({neg/len(labels):.1%})")
print(f"  Majority baseline: {max(pos,neg)/len(labels):.1%}")

## 6 — Train step-correctness probe

Trains a 3-layer MLP on `(h_c, label)` pairs.  
Target: val accuracy meaningfully above the majority baseline printed above.

In [ ]:
!python scripts/train_probe.py \
    --data    {PROBE_DATA_PATH} \
    --output  {PROBE_MODEL_PATH} \
    --epochs  {PROBE_EPOCHS} \
    --device  cuda

## 7 — Download results

Place files locally:
- `ssae_symbolic_p1.enc.pt` → `results/checkpoints/`
- `correctness_probe_symbolic.pt` → `results/probes/`
- `symbolic_ssae_p1.npz` → `results/probe_data/`

In [ ]:
from google.colab import files

files.download(ENC_PATH)          # SSAE encoder (~3 MB)
files.download(PROBE_MODEL_PATH)  # correctness probe (~6 MB)
files.download(PROBE_DATA_PATH)   # probe latents + labels .npz